# Mamba-3 Compatibility Test

**Goal:** Test if real Mamba-3 (from `state-spaces/mamba` main branch) can run on Colab A100.

| Step | What | Pass Criteria |
|------|------|---------------|
| 1 | Install from main branch | `from mamba_ssm import Mamba3` works |
| 2 | Forward pass | Output shape matches input |
| 3 | Backward pass | `.backward()` completes without crash |
| 4 | CrossScan integration | 3-axis scan with Mamba3 blocks |
| 5 | TextMamba3D smoke test | Full model forward+backward |

**Mamba-3 paper:** arxiv 2603.15569 (ICLR 2026)  
**Dependencies:** triton>=3.5.0, tilelang>=0.1.7.post3, nvidia-cutlass-dsl==4.4.1, quack-kernels==0.3.1

In [1]:
# Cell 1: Environment check
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

import sys
print(f'Python: {sys.version}')

# Check current triton version
try:
    import triton
    print(f'Triton: {triton.__version__}')
except ImportError:
    print('Triton: not installed')

Fri Mar 20 09:36:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             52W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Cell 2: Install real Mamba-3 from main branch
# CRITICAL: must uninstall PyPI version first, otherwise pip reuses cached 2.3.1

# Step 1: Uninstall existing mamba-ssm (PyPI v2.3.1 does NOT have Mamba3)
!pip uninstall mamba-ssm -y 2>/dev/null
!pip cache remove mamba_ssm 2>/dev/null

# Step 2: Install Mamba-3 dependencies (exact pinned versions from setup.py)
!pip install -q 'triton>=3.5.0' 'tilelang>=0.1.7.post3' 'nvidia-cutlass-dsl==4.4.1' 'quack-kernels==0.3.1'

# Step 3: Install mamba-ssm from main branch (has Mamba3)
!pip install -q --no-build-isolation --force-reinstall --no-deps \
    'mamba-ssm @ git+https://github.com/state-spaces/mamba.git@main'

# Step 4: Also install causal-conv1d (needed by Mamba1, optional for Mamba3)
!pip install -q 'causal-conv1d>=1.4.0'

# Verify installation
import importlib, mamba_ssm
importlib.reload(mamba_ssm)

print(f'mamba_ssm version: {mamba_ssm.__version__}')
print(f'mamba_ssm location: {mamba_ssm.__file__}')
print(f'Exports: {[x for x in dir(mamba_ssm) if x.startswith("Mamba")]}')

# The real test
from mamba_ssm import Mamba3
print(f'Mamba3 class: {Mamba3}')
print('OK - Mamba3 imported successfully')

In [ ]:
# Cell 2b: Diagnostic — only run if Cell 2 failed
# Checks whether Mamba3 code actually exists on the main branch

import os

try:
    from mamba_ssm import Mamba3
    print('Mamba3 already available, skip this cell')
except (ImportError, AttributeError):
    print('Mamba3 not available. Diagnosing...')

    # 1. Check what's installed
    !pip show mamba-ssm 2>/dev/null | grep -E "Version|Location"
    !python -c "import mamba_ssm; print('Location:', mamba_ssm.__file__); print('Dir:', [x for x in dir(mamba_ssm) if not x.startswith('_')])"

    # 2. Clone main branch and check if mamba3.py exists
    !rm -rf /tmp/mamba_check
    !git clone --depth 1 https://github.com/state-spaces/mamba.git /tmp/mamba_check

    mamba3_path = '/tmp/mamba_check/mamba_ssm/modules/mamba3.py'
    init_path = '/tmp/mamba_check/mamba_ssm/__init__.py'

    if os.path.exists(mamba3_path):
        size = os.path.getsize(mamba3_path)
        print(f'mamba3.py EXISTS on main ({size} bytes)')

        # Show class definition and __init__ signature
        print('--- Mamba3 class header ---')
        with open(mamba3_path) as f:
            in_class = False
            for i, line in enumerate(f):
                if 'class Mamba3' in line:
                    in_class = True
                if in_class:
                    print(f'{i+1}: {line.rstrip()}')
                    if line.strip().startswith('):') or (in_class and i > 120):
                        break
    else:
        print('mamba3.py NOT FOUND on main!')
        !ls /tmp/mamba_check/mamba_ssm/modules/

    # 3. Check __init__.py for Mamba3 export
    print('--- __init__.py Mamba exports ---')
    !grep -n 'Mamba3' /tmp/mamba_check/mamba_ssm/__init__.py || echo "Mamba3 NOT in __init__.py"

    # 4. Try editable install from cloned source
    print('--- Attempting editable install ---')
    !cd /tmp/mamba_check && pip install -e . --no-build-isolation 2>&1 | tail -10

    # 5. Re-test after editable install
    import importlib, mamba_ssm
    importlib.reload(mamba_ssm)
    try:
        from mamba_ssm import Mamba3
        print(f'SUCCESS after editable install: {Mamba3}')
    except Exception as e:
        print(f'Still failed: {e}')

In [ ]:
# Cell 3: Inspect Mamba3 API
# We need to know what constructor arguments Mamba3 accepts
# before wiring it into TextMamba3D

import inspect
from mamba_ssm import Mamba, Mamba2, Mamba3

print('=== Mamba (v1) ===')
print(inspect.signature(Mamba.__init__))
print()

print('=== Mamba2 (v2 SSD) ===')
print(inspect.signature(Mamba2.__init__))
print()

print('=== Mamba3 ===')
sig = inspect.signature(Mamba3.__init__)
print(sig)
print()

# List all parameters with defaults
print('Mamba3 parameters:')
for name, param in sig.parameters.items():
    if name == 'self':
        continue
    default = param.default if param.default != inspect.Parameter.empty else 'REQUIRED'
    print(f'  {name}: {default}')

In [ ]:
# Cell 4: Forward pass smoke test
import torch
from mamba_ssm import Mamba3

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats()

# Test with TextMamba3D-compatible dimensions
# Stage dims: 48, 96, 192, 384 (embed_dim=48, 4 stages)
test_configs = [
    {'d_model': 48,  'name': 'Stage 0 (48)'},
    {'d_model': 96,  'name': 'Stage 1 (96)'},
    {'d_model': 192, 'name': 'Stage 2 (192)'},
    {'d_model': 384, 'name': 'Stage 3 (384)'},
]

# Sequence lengths matching 3D cross-scan at each stage
# patch=4, img=128 -> spatial: 32,16,8,4
seq_lens = [32768, 4096, 512, 64]  # 32^3, 16^3, 8^3, 4^3

print('Forward pass test:')
for cfg, seq_len in zip(test_configs, seq_lens):
    try:
        # Try creating Mamba3 with d_state=16 (same as V4.5/V5.0)
        ssm = Mamba3(
            d_model=cfg['d_model'],
            d_state=16,
            expand=2,
        ).to(device)
        
        x = torch.randn(1, seq_len, cfg['d_model'], device=device)
        with torch.cuda.amp.autocast():
            out = ssm(x)
        
        assert out.shape == x.shape, f'Shape mismatch: {out.shape} != {x.shape}'
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f'  {cfg["name"]}: seq={seq_len} -> {out.shape} OK (peak {peak:.2f} GB)')
        
        del ssm, x, out
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    except Exception as e:
        print(f'  {cfg["name"]}: FAILED - {type(e).__name__}: {e}')

print('Forward pass test complete')

In [ ]:
# Cell 5: Backward pass test (this is where Triton kernels may crash)
import torch
from mamba_ssm import Mamba3

device = torch.device('cuda')

print('Backward pass test (Triton kernel stability):')
for d_model, seq_len, name in [
    (48, 512, 'Small (48, L=512)'),
    (96, 512, 'Medium (96, L=512)'),
    (192, 512, 'Large (192, L=512)'),
    (48, 4096, 'Long seq (48, L=4096)'),
    (96, 4096, 'Long seq (96, L=4096)'),
]:
    try:
        ssm = Mamba3(d_model=d_model, d_state=16, expand=2).to(device)
        x = torch.randn(2, seq_len, d_model, device=device, requires_grad=True)
        
        with torch.cuda.amp.autocast():
            out = ssm(x)
            loss = out.sum()
        
        loss.backward()  # <-- This is where illegal memory access may occur
        
        grad_norm = x.grad.norm().item()
        assert not torch.isnan(x.grad).any(), 'NaN in gradients!'
        assert not torch.isinf(x.grad).any(), 'Inf in gradients!'
        print(f'  {name}: OK (grad_norm={grad_norm:.4f})')
        
        del ssm, x, out, loss
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f'  {name}: FAILED - {type(e).__name__}: {e}')
        # If CUDA error, try to recover
        if 'CUDA' in str(e) or 'illegal' in str(e).lower():
            print('  -> CUDA error detected. This is the known Triton kernel bug.')
            print('  -> Remaining tests may be unreliable. Restarting runtime recommended.')
            break

print('Backward pass test complete')

In [ ]:
# Cell 6: Integration test with CrossScanBiMamba3DBlock
# This tests Mamba3 within the actual TextMamba3D pipeline

import os, sys
REPO_DIR = '/content/TextMamba3D'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    sys.path.insert(0, '.')
else:
    # If repo not cloned, clone it
    !git clone --depth 1 https://github.com/PlutoLei/TextMamba3D.git {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, '.')

from models.mamba_block import CrossScanBiMamba3DBlock, MAMBA3_AVAILABLE
print(f'MAMBA3_AVAILABLE: {MAMBA3_AVAILABLE}')

import torch
device = torch.device('cuda')

# Test CrossScan block with Mamba3 at stage 2 (dim=192, spatial=8x8x8)
print('CrossScanBiMamba3DBlock + Mamba3 test:')
try:
    block = CrossScanBiMamba3DBlock(
        dim=192,
        spatial_dims=(8, 8, 8),
        d_state=16,
        expand=2,
        use_mamba3=True,
    ).to(device)
    
    x = torch.randn(2, 512, 192, device=device, requires_grad=True)  # B=2, L=8^3=512
    
    with torch.cuda.amp.autocast():
        out = block(x)
        loss = out.sum()
    
    loss.backward()
    
    print(f'  Input:  {x.shape}')
    print(f'  Output: {out.shape}')
    print(f'  Grad:   {x.grad.norm().item():.4f}')
    print(f'  Peak:   {torch.cuda.max_memory_allocated()/1024**3:.2f} GB')
    print('  PASS')
    
    del block, x, out, loss
    torch.cuda.empty_cache()
    
except Exception as e:
    print(f'  FAILED: {type(e).__name__}: {e}')

In [ ]:
# Cell 7: Full model smoke test (2 samples, 1 forward+backward)
import os, sys, yaml, torch
os.chdir(REPO_DIR)
sys.path.insert(0, '.')

from models.textmamba3d import TextMamba3D

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats()

print('Full TextMamba3D + Mamba3 smoke test:')
try:
    model = TextMamba3D(
        img_size=(128, 128, 128),
        in_channels=4,
        out_channels=4,
        embed_dim=48,
        depths=[2, 2, 2, 2],
        d_state=16,
        text_embed_dim=256,
        text_max_len=192,
        use_mamba3=True,
        fusion_type='seqca',
        deep_supervision=True,
    ).to(device)
    
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {params:,}')
    
    # Forward with text
    img = torch.randn(1, 4, 128, 128, 128, device=device)
    text_ids = torch.randint(0, 30000, (1, 64), device=device)
    attn_mask = torch.ones(1, 64, device=device)
    
    with torch.cuda.amp.autocast():
        out = model(img, text_ids, attn_mask)
        if isinstance(out, (list, tuple)):
            loss = out[0].sum()
        else:
            loss = out.sum()
    
    loss.backward()
    
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  Output shape: {out[0].shape if isinstance(out, (list,tuple)) else out.shape}')
    print(f'  Peak GPU: {peak:.1f} / {total:.0f} GB')
    print(f'  PASS - Mamba3 full pipeline works!')
    
    del model, img, text_ids, attn_mask, out, loss
    torch.cuda.empty_cache()

except Exception as e:
    print(f'  FAILED: {type(e).__name__}: {e}')
    import traceback
    traceback.print_exc()

## Results Summary

Fill in after running:

| Test | Status | Notes |
|------|--------|-------|
| Install | | |
| Forward pass | | |
| Backward pass | | |
| CrossScan integration | | |
| Full model | | |

If all pass: proceed to create `TextMamba3D_A100_V5.1.ipynb` (real Mamba-3 training)  
If backward fails: report specific error for upstream fix